# Smart City Autonomous Driving — Traffic FSM & Obstacle Evasion

This notebook demonstrates autonomous urban driving for JetRacer. It integrates object detection, traffic state machine logic, road obstacle safety evaluation, and hardware motor control.

### System Component Overview:

| Component | Module | High-Level Function |
| :--- | :--- | :--- |
| **Traffic Sign Model** | `YOLOProcessor` | Detects traffic lights and directional navigation signs. |
| **Road Status Model** | `RoadProcessor` | Evaluates road status to classify whether the path ahead is clear or blocked. |
| **Traffic FSM** | `TrafficFSM` | Filters detections spatially and manages traffic rule priorities. |
| **Decision Engine** | `IntersectionDecisionMaker` | Cross-references sign intent with road availability to execute intersection maneuvers. |
| **Vehicle Controller** | `RacecarController` | Hardware abstraction layer managing steering, throttle, and turn maneuvers. |

---

## Section 1: Setup & Dependencies

Load core package dependencies including hardware drivers, image processors, and traffic state machine components.

In [ ]:
# %load_ext autoreload
%autoreload 2


In [ ]:
# import os
# sim_path = os.path.abspath(os.path.join("..", "src", "simulation"))
# if sim_path not in sys.path:


## Section 2: Hardware Acceleration

Verify available ONNX Runtime execution providers to enable GPU acceleration when running on Jetson Nano hardware.

In [ ]:
import sys
import os
import cv2
import numpy as np
import onnxruntime as ort
import rospy
from sensor_msgs.msg import Image
from jetracer_ai.hardware import NvidiaRacecar, RacecarController

print(f"[+] ONNX Runtime Version: {ort.__version__}")

# Check execution providers (Prefer CUDA/TensorRT)
available_providers = ort.get_available_providers()
print(f"[+] Available Providers: {available_providers}")

if 'CUDAExecutionProvider' in available_providers:
    print("[✓] ONNX Runtime ready for GPU (CUDA) execution!")
else:
    print("[!] Warning: CUDA Provider not found. Model will run on CPU.")


## Section 3: Model Loading & Processors

Load trained ONNX AI models for traffic sign detection and road safety classification, and initialize input image processors.

In [ ]:
import gc
import time
import cv2
import numpy as np
import rospy
from IPython.display import Image, clear_output, display

from jetracer_ai.utils import CameraStream 
from jetracer_ai.urban_traffic import YOLOProcessor, RoadProcessor
from jetracer_ai.urban_traffic import TrafficFSM
from jetracer_ai.core import ONNXEngine
from jetracer_ai.hardware import RacecarController

# Initialize Camera Stream
cam = CameraStream(
    topic_name='/csi_cam_0/image_raw', width=500, height=300, record_video=False
)

# 1. Initialize ONNX Runtime Engine
traffic_engine = ONNXEngine('../models/best.onnx')
road_engine = ONNXEngine('../models/best_model_mobilenet.onnx')

# 2. Initialize Individual Processors
traffic_processor = YOLOProcessor(img_size=(640))
road_processor = RoadProcessor(img_size=(224, 224), threshold=0.5)

print("✅ Successfully loaded ONNX Models and initialized Processors!")


## Section 4: Traffic FSM & Controller Configuration

Initialize the traffic state machine with spatial filtering rules and configure the vehicle motor controller.

In [ ]:
from jetracer_ai.hardware import RacecarController

car = RacecarController(base_throttle=0., max_throttle=0.4)


## Section 5: Real-Time Autonomous Control Loop

Execute the main autonomous control loop. Captures camera frames, processes sign detections, checks road safety, updates vehicle control actions, and displays telemetry log overlays.

In [ ]:
DISPLAY_UI, SKIP_FRAMES, TARGET_FPS = True, 1, 20

# ----------------------------------------------------
# 1. KHỞI TẠO STATE MACHINE NÉ VẬT CẢN (ESCAPE FSM)
# ----------------------------------------------------
CONFIRM_FRAMES = 3
CONFIRM_THRESHOLD = 0.5  # Threshold to determine blocked road status

maneuver_state = 'DRIVE'
blocked_frame_count = 0
maneuver_start_time = time.time()

# ----------------------------------------------------
# 2. KHỞI TẠO TRAFFIC FSM & CAMERA / ROS
# ----------------------------------------------------
fsm = TrafficFSM(
    default_state='FORWARD',
    conf_threshold=0.5,
    min_consecutive_frames=3,
    state_timeout=2.0,
    min_bbox_area=900,
    roi_x_min=0.15,
    roi_x_max=0.85,
)

display_handle = display(None, display_id=True) if DISPLAY_UI else None
rate = rospy.Rate(TARGET_FPS)
frame_count = 0

rospy.loginfo("🚀 Starting Debug Stream + Escape Maneuver + Auto Control Loop...")
# Constant definitions for UI logging
# Width parameter for right-side dashboard log area
LOG_PANEL_WIDTH = 350 

# Variable tracking for actual throttle and steering values
actual_steering = 0.0
actual_throttle = 0.0

# --- START AUTONOMOUS MAIN LOOP ---
try:
    while not rospy.is_shutdown():
        frame = cam.get_frame()
        if frame is None:
            rospy.logwarn_throttle(2.0, "⏳ Waiting for camera frames...")
            rate.sleep()
            continue

        start_time, frame_count = rospy.get_time(), frame_count + 1
        now = time.time()

        # --- 1. AI INFERENCE ---
        t_input, orig_h, orig_w = traffic_processor.preprocess(frame)
        detections = traffic_processor.postprocess(traffic_engine.infer(t_input), orig_h, orig_w)

        r_input = road_processor.preprocess(frame)
        road_result = road_processor.postprocess(road_engine.infer(r_input))

        road_status, blocked_prob = road_result['status'], road_result['blocked_probability']

        # --- 2. ROAD OBSTACLE FRAME DEBOUNCING ---
        if blocked_prob >= CONFIRM_THRESHOLD:
            blocked_frame_count += 1
        else:
            blocked_frame_count = max(0, blocked_frame_count - 1)

        # --- 3. OBSTACLE EVASION FSM & MOTION CONTROL UPDATE ---
        if maneuver_state == 'DRIVE':
            if blocked_frame_count >= CONFIRM_FRAMES:
                maneuver_state = 'REVERSE_TURNING'
                maneuver_start_time = now
                # Update display status values
                actual_steering = -0.7
                actual_throttle = -0.22
                # Execute vehicle motion control
                car.steering = actual_steering
                car.throttle = actual_throttle
            else:
                fsm_action = fsm.update(detections, img_w=orig_w, img_h=orig_h)
                # Compute steer and throttle based on current FSM state 
                # and execute control action on vehicle controller 
                # Update car steering and throttle values
                car.execute_action(fsm_action)
                actual_steering = car.steering
                actual_throttle = car.throttle

        elif maneuver_state == 'REVERSE_TURNING':
            if now - maneuver_start_time < 1.2:
                actual_steering = -0.7
                actual_throttle = -0.22
            else:
                maneuver_state = 'PAUSE'
                maneuver_start_time = now
                actual_steering = 0.0
                actual_throttle = 0.0
            car.steering = actual_steering
            car.throttle = actual_throttle

        elif maneuver_state == 'PAUSE':
            if now - maneuver_start_time < 0.3:
                actual_steering = 0.0
                actual_throttle = 0.0
            else:
                maneuver_state = 'CHECK_FORWARD'
                maneuver_start_time = now
                actual_steering = 0.0
                actual_throttle = 0.15
            car.steering = actual_steering
            car.throttle = actual_throttle

        elif maneuver_state == 'CHECK_FORWARD':
            if now - maneuver_start_time < 0.8:
                actual_steering = 0.0
                actual_throttle = 0.15
            else:
                if blocked_frame_count < CONFIRM_FRAMES:
                    maneuver_state = 'DRIVE'
                    blocked_frame_count = 0
                    # Value will be computed in subsequent iteration
                else:
                    maneuver_state = 'REVERSE_TURNING'
                    maneuver_start_time = now
                    actual_steering = -0.7
                    actual_throttle = -0.22
            car.steering = actual_steering
            car.throttle = actual_throttle

        # --- 4. ROS LOGGING & METRICS ---
        latency_ms = (rospy.get_time() - start_time) * 1000
        fps_real = 1000 / max(latency_ms, 1)

        if frame_count % 50 == 0:
            clear_output(wait=True)
            if DISPLAY_UI:
                display_handle = display(None, display_id=True)

        rospy.loginfo_throttle(
            1.0, 
            f"🛣️ Road: [{road_status}] | 🔄 Maneuver: [{maneuver_state}] | ☸️ S:{actual_steering:.2f} T:{actual_throttle:.2f} | FPS: {fps_real:.1f}"
        )

        # --- 5. UI DASHBOARD DISPLAY OVERLAY ---
        if DISPLAY_UI and (frame_count % SKIP_FRAMES == 0):
            # 5a. Draw Bounding Boxes on main camera frame
            debug_frame = traffic_processor.draw_bboxes(frame, detections)
            
            # --- 5b. CREATE RIGHT DASHBOARD LOG PANEL ---
            # Create black background dashboard panel matching frame height
            h, w, c = debug_frame.shape
            log_panel = np.zeros((h, LOG_PANEL_WIDTH, c), dtype=np.uint8)
            
            # --- 5c. RENDER TEXT METRICS ON DASHBOARD PANEL ---
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.5
            font_thickness = 1
            text_color = (255, 255, 255) # White color
            status_color = (0, 0, 255) if maneuver_state != 'DRIVE' else (0, 255, 0) # Red if stopped, Green if driving
            
            y_offset = 30
            line_height = 25
            
            # Dashboard Title
            cv2.putText(log_panel, "=== CAR DASHBOARD ===", (10, y_offset), font, 0.6, (0, 255, 255), 2)
            y_offset += 40
            
            # Maneuver State Display
            cv2.putText(log_panel, f"Maneuver: ", (10, y_offset), font, font_scale, text_color, font_thickness)
            status_color = (0, 0, 255) if maneuver_state != 'DRIVE' else (0, 255, 0) # Red if stopped, Green if driving
            y_offset += line_height
            
            # Road Safety Status
            cv2.putText(log_panel, f"Road: {road_status} ({blocked_prob:.2f})", (10, y_offset), font, font_scale, text_color, font_thickness)
            y_offset += line_height
            
            # Visual Section Divider
            cv2.putText(log_panel, "--- Controls ---", (10, y_offset), font, font_scale, (150, 150, 150), font_thickness)
            y_offset += line_height
            
            # STEERING AND THROTTLE CONTROL VALUES
            steer_text = f"Steering: {actual_steering:.2f}"
            throttle_text = f"Throttle: {actual_throttle:.2f}"
            cv2.putText(log_panel, steer_text, (20, y_offset), font, font_scale, text_color, font_thickness)
            y_offset += line_height
            cv2.putText(log_panel, throttle_text, (20, y_offset), font, font_scale, text_color, font_thickness)
            y_offset += 40 # Panel row vertical offset
            
            # FPS and Performance Metrics
            cv2.putText(log_panel, "--- System ---", (10, y_offset), font, font_scale, (150, 150, 150), font_thickness)
            y_offset += line_height
            cv2.putText(log_panel, f"FPS: {fps_real:.1f}", (10, y_offset), font, font_scale, text_color, font_thickness)
            y_offset += line_height
            cv2.putText(log_panel, f"Frame: {frame_count}", (10, y_offset), font, font_scale, text_color, font_thickness)
            y_offset += line_height
            cv2.putText(log_panel, f"Latency: {latency_ms:.1f} ms", (10, y_offset), font, font_scale, text_color, font_thickness)
            
            # Display Detected Traffic Signs
            if len(detections) > 0:
                y_offset += 40
                cv2.putText(log_panel, "--- Detections ---", (10, y_offset), font, font_scale, (150, 150, 150), font_thickness)
                y_offset += line_height
                for i, d in enumerate(detections[:5]): # Display up to 5 detected signs
                    det_str = f"- {d['class_name']} ({d['confidence']:.2f})"
                    cv2.putText(log_panel, det_str, (10, y_offset), font, 0.4, (0, 200, 0), font_thickness)
                    y_offset += 20

            # --- 5d. CONCATENATE CAMERA FEED AND LOG PANEL ---
            # Combine camera view and status dashboard horizontally
            final_display_frame = np.hstack((debug_frame, log_panel))
            
            # --- 5e. UPDATE JUPYTER UI DISPLAY ---
            rgb_frame = cv2.cvtColor(final_display_frame, cv2.COLOR_BGR2RGB)
            _, jpeg = cv2.imencode('.jpg', rgb_frame, [int(cv2.IMWRITE_JPEG_QUALITY), 70])
            display_handle.update(Image(data=jpeg.tobytes()))

        if frame_count % 100 == 0:
            gc.collect()
            
        rate.sleep()

except KeyboardInterrupt:
    car.stop()
    rospy.loginfo("🛑 Vehicle stopped by KeyboardInterrupt.")
finally:
    car.stop()
    rospy.loginfo("🛑 Program terminated cleanly, vehicle safely stopped.")
